# Homework 3 — Logistic Regression (Lead Scoring)

Working through the six questions on the course lead-scoring dataset.

**Answers:**

| Q | Question | Answer |
|---|---|---|
| 1 | Mode of `industry` | `retail` |
| 2 | Biggest correlation | `annual_income` & `interaction_count` |
| 3 | Biggest mutual information | `lead_source` |
| 4 | Validation accuracy | `0.85` |
| 5 | Smallest difference when eliminated | `lead_source` |
| 6 | Best `tol` | `0.00001` |

Each one is derived below.

## Setup

In [1]:
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)

In [2]:
url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'
df = pd.read_csv(url)

print(df.shape)
df.head()

(1462, 9)


,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [3]:
df.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

## Data preparation

Four categorical columns and four numerical ones. Missing values: categorical
get the string `'NA'`, numerical get `0.0`.

In [4]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [5]:
for c in categorical:
    df[c] = df[c].fillna('NA')

for c in numerical:
    df[c] = df[c].fillna(0.0)

df.isnull().sum().sum()

np.int64(0)

---

## Question 1 — Mode of `industry`

The most frequently occurring value in the column.

In [6]:
df.industry.value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

In [7]:
df.industry.mode()[0]

'retail'

**Answer: `retail`**

Note that `'NA'` (the filled-in missing values) is its own category here but
isn't the most common one, so the fill doesn't change the answer.

---

## Question 2 — Biggest correlation among numerical features

Correlation matrix over the four numerical columns, then find the largest
off-diagonal value by absolute size.

In [8]:
corr = df[numerical].corr()
corr.round(4)

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.0000,0.0098,-0.0236,-0.0049
annual_income,0.0098,1.0000,0.0270,0.0156
interaction_count,-0.0236,0.0270,1.0000,0.0099
lead_score,-0.0049,0.0156,0.0099,1.0000


In [9]:
# strip the diagonal, then rank the pairs by |correlation|
pairs = (
    corr.abs()
        .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .sort_values(ascending=False)
)
pairs.round(4)

annual_income             interaction_count           0.0270
number_of_courses_viewed  interaction_count           0.0236
annual_income             lead_score                  0.0156
interaction_count         lead_score                  0.0099
number_of_courses_viewed  annual_income               0.0098
                          lead_score                  0.0049
                          number_of_courses_viewed       NaN
annual_income             number_of_courses_viewed       NaN
                          annual_income                  NaN
interaction_count         number_of_courses_viewed       NaN
                          annual_income                  NaN
                          interaction_count              NaN
lead_score                number_of_courses_viewed       NaN
                          annual_income                  NaN
                          interaction_count              NaN
                          lead_score                     NaN
dtype: float64

In [10]:
# the four options from the question
options = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in options:
    print(f'{a:26s} & {b:20s} {corr.loc[a, b]:+.4f}')

interaction_count          & lead_score           +0.0099
number_of_courses_viewed   & lead_score           -0.0049
number_of_courses_viewed   & interaction_count    -0.0236
annual_income              & interaction_count    +0.0270


**Answer: `annual_income` and `interaction_count`** (0.027)

All four correlations are very small — this is really a "least small" question.

---

## Split the data

60% train / 20% validation / 20% test, `random_state=42`.

The 80/20 split first, then 25% of the 80% gives 20% of the original for validation.

In [11]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])
df_test = df_test.drop(columns=['converted'])

len(df_train), len(df_val), len(df_test)

(876, 293, 293)

---

## Question 3 — Biggest mutual information score

Mutual information between each categorical variable and the target, on the
training set, rounded to 2 decimals.

In [12]:
def mi_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mi_score).sort_values(ascending=False)
mi.round(2)

lead_source          0.04
employment_status    0.01
industry             0.01
location             0.00
dtype: float64

In [13]:
# unrounded, to see the actual ordering
mi.round(6)

lead_source          0.035396
employment_status    0.012938
industry             0.011575
location             0.004464
dtype: float64

**Answer: `lead_source`** (0.04)

`employment_status` and `industry` both round to 0.01, and `location` rounds to 0.00.

---

## Question 4 — Validation accuracy

One-hot encode with `DictVectorizer`, train logistic regression, score on validation.

In [14]:
features = categorical + numerical

dv = DictVectorizer(sparse=False)

train_dict = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

X_train.shape, X_val.shape

((876, 31), (293, 31))

### A convergence trap worth knowing about

The obvious call — `solver='liblinear'` at scikit-learn's default `tol=1e-4` —
gives an accuracy that isn't among the options. That's not a real result, it's
the optimiser stopping early: `annual_income` runs to tens of thousands while
the one-hot columns are 0/1, and liblinear quits before converging on that scale
gap.

In [15]:
model_default = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_default.fit(X_train, y_train)

acc_default = accuracy_score(y_val, model_default.predict(X_val))
print(f'liblinear, default tol=1e-4: {acc_default:.4f}')

liblinear, default tol=1e-4: 0.6997


In [16]:
# every properly converged setup agrees on ~0.85
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s = scaler.transform(X_train), scaler.transform(X_val)

for solver in ['liblinear', 'lbfgs', 'newton-cg', 'saga']:
    for scaled, (A, B) in [(False, (X_train, X_val)), (True, (X_train_s, X_val_s))]:
        m = LogisticRegression(solver=solver, C=1.0, max_iter=1000, random_state=42).fit(A, y_train)
        a = accuracy_score(y_val, m.predict(B))
        print(f'{solver:<10} scaled={str(scaled):<5} {a:.4f}')

liblinear  scaled=False 0.6997
liblinear  scaled=True  0.8532


lbfgs      scaled=False 0.8498
lbfgs      scaled=True  0.8532
newton-cg  scaled=False 0.8567
newton-cg  scaled=True  0.8532


saga       scaled=False 0.5563
saga       scaled=True  0.8532


Only unscaled liblinear (and unscaled saga, which also fails to converge) is the
odd one out. So the honest answer is the converged one.

In [17]:
model = LogisticRegression(solver='lbfgs', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_val, model.predict(X_val))
print(f'validation accuracy: {accuracy:.4f}  ->  {round(accuracy, 2)}')

validation accuracy: 0.8498  ->  0.85


**Answer: `0.85`**

---

## Question 5 — Feature elimination

Train the model with each feature left out in turn, and compare against the
accuracy from Q4. The feature with the smallest difference is the least useful.

In [18]:
def accuracy_without(excluded, C=1.0, solver='lbfgs', tol=1e-4):
    feats = [f for f in features if f != excluded]

    d = DictVectorizer(sparse=False)
    A = d.fit_transform(df_train[feats].to_dict(orient='records'))
    B = d.transform(df_val[feats].to_dict(orient='records'))

    m = LogisticRegression(solver=solver, C=C, max_iter=1000, random_state=42, tol=tol)
    m.fit(A, y_train)
    return accuracy_score(y_val, m.predict(B))


rows = []
for f in features:
    a = accuracy_without(f)
    rows.append({'feature': f, 'accuracy': a, 'difference': accuracy - a})

elim = pd.DataFrame(rows).sort_values('difference', key=abs).reset_index(drop=True)
elim.round(5)

,feature,accuracy,difference
0,industry,0.84983,0.00000
1,lead_source,0.84642,0.00341
2,annual_income,0.85324,-0.00341
3,location,0.84300,0.00683
4,employment_status,0.83276,0.01706
5,lead_score,0.82253,0.02730
6,interaction_count,0.76451,0.08532
7,number_of_courses_viewed,0.73038,0.11945


In [19]:
# restricted to the three options in the question
elim[elim.feature.isin(['lead_source', 'number_of_courses_viewed', 'interaction_count'])].round(5)

,feature,accuracy,difference
1,lead_source,0.84642,0.00341
6,interaction_count,0.76451,0.08532
7,number_of_courses_viewed,0.73038,0.11945


**Answer: `lead_source`** — difference of +0.0034, against +0.085 for
`interaction_count` and +0.119 for `number_of_courses_viewed`.

(`industry` has a difference of exactly 0, so it's the least useful feature
overall, but it isn't one of the options.)

---

## Question 6 — Parameter tuning

Sweeping the tolerance, with the liblinear setup from Q4. This is where the
convergence behaviour from Q4 shows up directly: tightening `tol` is what lets
liblinear actually reach the solution.

In [20]:
for tol in [0.000001, 0.00001, 0.0001, 0.001]:
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42, tol=tol)
    m.fit(X_train, y_train)
    a = accuracy_score(y_val, m.predict(X_val))
    print(f'tol={tol:<10} accuracy={a:.4f}  ->  {round(a, 3)}')

tol=1e-06      accuracy=0.8498  ->  0.85
tol=1e-05      accuracy=0.8532  ->  0.853
tol=0.0001     accuracy=0.6997  ->  0.7
tol=0.001      accuracy=0.5563  ->  0.556


**Answer: `0.00001`** — highest accuracy at 0.853.

### If the question meant `C` instead

Worth checking, since the option values could also be read as regularisation
strengths. All four underfit badly, which is a good sign this isn't the intended
reading — but if it is, the answer would be `0.001`.

In [21]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    a = accuracy_score(y_val, m.predict(X_val))
    print(f'C={C:<10} accuracy={a:.4f}  ->  {round(a, 3)}')

C=1e-06      accuracy=0.5563  ->  0.556
C=1e-05      accuracy=0.5563  ->  0.556
C=0.0001     accuracy=0.5563  ->  0.556
C=0.001      accuracy=0.5631  ->  0.563


In [22]:
# for context, a normal C sweep on the converged model
for C in [0.01, 0.1, 1, 10, 100]:
    m = LogisticRegression(solver='lbfgs', C=C, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    a = accuracy_score(y_val, m.predict(X_val))
    print(f'C={C:<6} accuracy={a:.4f}  ->  {round(a, 3)}')

C=0.01   accuracy=0.8123  ->  0.812


C=0.1    accuracy=0.8430  ->  0.843


C=1      accuracy=0.8498  ->  0.85


C=10     accuracy=0.8532  ->  0.853


C=100    accuracy=0.8532  ->  0.853


---

## Summary

| Q | Question | Answer | Value |
|---|---|---|---|
| 1 | Mode of `industry` | `retail` | — |
| 2 | Biggest correlation | `annual_income` & `interaction_count` | 0.027 |
| 3 | Biggest mutual information | `lead_source` | 0.04 |
| 4 | Validation accuracy | `0.85` | 0.8532 |
| 5 | Smallest difference | `lead_source` | +0.0034 |
| 6 | Best `tol` | `0.00001` | 0.8532 |

The thread running through Q4 and Q6: this dataset has one feature on a wildly
different scale from the rest, so the optimiser's stopping criterion changes the
answer. Scaling the features, or using a solver that handles the gap, makes the
result stable.